# DermKG visualization: from one node to the whole graph

A zoom sequence over the knowledge graph built by `dermkg_build.ipynb`, one figure per cell:

1. **One node**: a disease and its direct neighbors
2. **Two hops**: the disease's knowledge neighborhood (concept groups, rule premises, category hierarchy)
3. **One case**: a single image's reasoning chain, image to concepts to rules to disease to category
4. **One category**: all diseases of a category with their concepts and rules
5. **One super-category**: the malignant branch
6. **Knowledge layer**: all non-image nodes in a radial layout
7. **Whole graph**: all 3,452 nodes including images
8. **Zoom strip**: panels 1, 3, 5 and 6 side by side for the paper

Static figures are saved as PDF and PNG (`figures_kg/`); each cell can also export an interactive HTML version (`EXPORT_HTML = True`).

## Setup

In [ ]:
KG_DIR = "data/dermkg"
FIG_DIR = "figures_kg"
FORMATS = ("pdf", "png")
EXPORT_HTML = True                     # also write interactive pyvis HTML for each subgraph
FOCUS_DISEASE = "basal cell carcinoma"  # node for panels 1 and 2
FOCUS_CATEGORY = "malignant epidermal"
FOCUS_SUPER = "malignant"
SEED = 7

In [ ]:
import os, json, ast
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib as mpl
import matplotlib.pyplot as plt

os.makedirs(FIG_DIR, exist_ok=True)
SINGLE, DOUBLE = 3.5, 7.16
mpl.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "Times", "DejaVu Serif"], "font.size": 7,
                     "legend.fontsize": 6.5, "legend.frameon": False, "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
                     "pdf.fonttype": 42, "ps.fonttype": 42})
COL = {"SkinImage": "#BBBBBB", "Concept": "#0072B2", "ConceptGroup": "#56B4E9", "Disease": "#D55E00",
       "DiseaseCategory": "#E69F00", "SuperCategory": "#CC79A7", "SkinType": "#F0E442", "ClinicalRule": "#009E73"}
SIZE = {"SkinImage": 6, "Concept": 60, "ConceptGroup": 90, "Disease": 60, "DiseaseCategory": 140, "SuperCategory": 200,
        "SkinType": 40, "ClinicalRule": 45}
SHAPE = {"ClinicalRule": "D", "SkinImage": "o", "SkinType": "s"}
ECOL = {"HAS_CONCEPT": "#9ecae1", "HAS_SKIN_TYPE": "#e5d96b", "DIAGNOSED_AS": "#f4a582", "BELONGS_TO": "#56B4E9",
        "IS_A": "#E69F00", "SUBCLASS_OF": "#CC79A7", "CHARACTERIZED_BY": "#D55E00", "CO_OCCURS": "#0072B2",
        "GOVERNED_BY": "#66c2a4", "INDICATES": "#009E73", "ACTIVATES": "#a1d99b"}
CAPTIONS = {}


def save(fig, name, caption, H=None):
    for ext in FORMATS:
        fig.savefig(os.path.join(FIG_DIR, f"{name}.{ext}"), dpi=300)
    if EXPORT_HTML and H is not None:
        to_html(H, os.path.join(FIG_DIR, f"{name}.html"))
    CAPTIONS[name] = caption
    with open(os.path.join(FIG_DIR, "captions.md"), "w") as f:
        for k in sorted(CAPTIONS):
            f.write(f"**{k}**: {CAPTIONS[k]}\n\n")
    plt.show()
    print(f"[{name}] {caption}")

In [ ]:
nodes = pd.read_csv(os.path.join(KG_DIR, "nodes.csv"))
tri = pd.read_csv(os.path.join(KG_DIR, "triples_full.csv"))
rules = pd.read_csv(os.path.join(KG_DIR, "clinical_rules.csv"))
rules["premise"] = rules["premise"].map(json.loads); rules["targets"] = rules["targets"].map(ast.literal_eval)
labels = pd.read_csv(os.path.join(KG_DIR, "labels.csv"))

G = nx.MultiDiGraph()
for r in nodes.to_dict("records"):
    G.add_node(r["id"], **{k: v for k, v in r.items() if k != "id" and pd.notna(v)})
for h, rel, t, w, sp in tri[["head", "relation", "tail", "weight", "split"]].itertuples(index=False):
    G.add_edge(h, t, rel=rel, weight=float(w), split=sp if isinstance(sp, str) else "")
NT = nx.get_node_attributes(G, "ntype")
print(G, "|", pd.Series(NT).value_counts().to_dict())


def short(n):
    name = str(G.nodes[n].get("name", n.split(":", 1)[-1]))
    name = name.replace("(Hyperpigmentation)", "").replace("(Hypopigmentation)", "")
    if NT[n] == "ClinicalRule":
        return n.split(":", 1)[1]
    return name


def sub(nodeset, rels=None):
    H = nx.MultiDiGraph()
    H.add_nodes_from((n, G.nodes[n]) for n in nodeset)
    for u, v, a in G.edges(data=True):
        if u in H and v in H and (rels is None or a["rel"] in rels):
            H.add_edge(u, v, **a)
    return H


def draw(H, pos, ax, label_types=("Concept", "Disease", "DiseaseCategory", "SuperCategory", "ClinicalRule", "ConceptGroup", "SkinType"),
         size_scale=1.0, edge_alpha=0.6, width_scale=1.0, fontsize=6, label_offset=None, highlight=None):
    for u, v, a in H.edges(data=True):
        (x1, y1), (x2, y2) = pos[u], pos[v]
        w = (0.3 + 1.6 * a["weight"]) if a["rel"] == "CHARACTERIZED_BY" else 0.6
        ax.plot([x1, x2], [y1, y2], color=ECOL[a["rel"]], lw=w * width_scale, alpha=edge_alpha, zorder=1,
                solid_capstyle="round")
    for t in COL:
        ns = [n for n in H if NT[n] == t]
        if not ns:
            continue
        xy = np.array([pos[n] for n in ns])
        s = [SIZE[t] * size_scale * (2.2 if highlight and n in highlight else 1) for n in ns]
        ax.scatter(xy[:, 0], xy[:, 1], s=s, c=COL[t], marker=SHAPE.get(t, "o"), edgecolors="white",
                   linewidths=0.4, zorder=3 if t != "SkinImage" else 2)
    for n in H:
        if label_types and NT[n] in label_types:
            dx, dy = (label_offset(n) if label_offset else (0, 0.035))
            ax.text(pos[n][0] + dx, pos[n][1] + dy, short(n), fontsize=fontsize, ha="center", va="bottom", zorder=4,
                    bbox=dict(boxstyle="round,pad=0.05", fc="white", ec="none", alpha=0.7))
    ax.set_aspect("equal"); ax.axis("off")


def legend(ax, types, rels=(), loc="lower center", anchor=(0.5, -0.02), ncol=4):
    h = [mpl.lines.Line2D([], [], marker=SHAPE.get(t, "o"), ls="", color=COL[t], markersize=5, label=t) for t in types]
    h += [mpl.lines.Line2D([], [], color=ECOL[r], lw=1.2, label=r) for r in rels]
    ax.legend(handles=h, loc=loc, bbox_to_anchor=anchor, ncol=ncol, fontsize=5.5, handletextpad=0.3, columnspacing=0.8)


def to_html(H, path):
    try:
        from pyvis.network import Network
    except ImportError:
        print("pyvis not installed, HTML skipped"); return
    net = Network(height="800px", width="100%", directed=True, cdn_resources="in_line")
    for n, a in H.nodes(data=True):
        net.add_node(n, label=short(n) if NT[n] != "SkinImage" else "", color=COL[NT[n]], title=f"{NT[n]}: {a.get('name', n)}",
                     size=max(4, SIZE[NT[n]] / 6), shape="diamond" if NT[n] == "ClinicalRule" else "dot")
    for u, v, a in H.edges(data=True):
        net.add_edge(u, v, title=a["rel"], color=ECOL[a["rel"]], width=1 + 3 * a["weight"] if a["rel"] == "CHARACTERIZED_BY" else 1)
    net.toggle_physics(H.number_of_nodes() < 600)
    net.save_graph(path)


def ring(names, r, start=0.0, end=2 * np.pi, center=(0, 0)):
    k = len(names)
    ang = np.linspace(start, end, k, endpoint=(end - start) < 2 * np.pi - 1e-6) if k > 1 else np.array([(start + end) / 2])
    if (end - start) >= 2 * np.pi - 1e-6:
        ang = start + 2 * np.pi * np.arange(k) / max(k, 1)
    return {n: (center[0] + r * np.cos(a), center[1] + r * np.sin(a)) for n, a in zip(names, ang)}

def radial_label(ax, p, text, center=(0, 0), off=0.07, fs=6, color="black", **kw):
    v = np.array(p) - np.array(center); a = np.arctan2(v[1], v[0]); c, sn = np.cos(a), np.sin(a)
    ha = "left" if c > 0.15 else "right" if c < -0.15 else "center"
    va = "center" if abs(c) > 0.15 else ("bottom" if sn > 0 else "top")
    ax.text(p[0] + off * c, p[1] + off * sn, text, fontsize=fs, ha=ha, va=va, color=color, zorder=5,
            bbox=dict(boxstyle="round,pad=0.04", fc="white", ec="none", alpha=0.75), **kw)


def layered(H, layers, sweeps=4, xgap=1.0, height=1.0, fixed=(), spacing=None):
    """layered layout with barycenter ordering to reduce edge crossings"""
    layers = [list(L) for L in layers]
    U = nx.Graph(H)
    for _ in range(sweeps):
        for i in list(range(1, len(layers))) + list(range(len(layers) - 2, -1, -1)):
            ref = {}
            for j in (i - 1, i + 1):
                if 0 <= j < len(layers):
                    for k, n in enumerate(layers[j]):
                        ref[n] = k / max(1, len(layers[j]) - 1)
            def bc(n):
                nb = [ref[x] for x in U.neighbors(n) if x in ref]
                return np.mean(nb) if nb else 0.5
            if i not in fixed:
                layers[i].sort(key=bc)
    pos = {}
    for i, L in enumerate(layers):
        if spacing:
            ys = -(np.arange(len(L)) - (len(L) - 1) / 2) * spacing
        else:
            ys = np.linspace(height / 2, -height / 2, len(L)) if len(L) > 1 else [0.0]
        pos.update({n: (i * xgap, y) for n, y in zip(L, ys)})
    return pos


def sector_positions(radii, gap=2):
    """radial layout of the knowledge layer: diseases in category sectors; other nodes at the mean angle of their diseases"""
    sups = sorted(n for n in G if NT[n] == "SuperCategory")
    sup_of = {c: v for c in G if NT[c] == "DiseaseCategory" for _, v, a in G.out_edges(c, data=True) if a["rel"] == "SUBCLASS_OF"}
    cats = sorted(sup_of, key=lambda c: (sup_of[c], c))
    slots = []
    for c in cats:
        ds = sorted(x for x in G if NT[x] == "Disease" and f"Category:{G.nodes[x]['category']}" == c)
        slots += ds + [None] * gap
    ang = {n: 2 * np.pi * k / len(slots) for k, n in enumerate(slots) if n}
    P = {n: (radii["Disease"] * np.cos(a), radii["Disease"] * np.sin(a)) for n, a in ang.items()}
    def cm(angles):
        a = np.array(angles); return np.arctan2(np.sin(a).mean(), np.cos(a).mean())
    cat_ang = {c: cm([ang[x] for x in ang if G.has_edge(x, c)]) for c in cats}
    off = cat_ang[cats[0]]
    for k, c in enumerate(cats):     # evenly spaced in sector order, so small categories do not overlap
        a = off + 2 * np.pi * k / len(cats); P[c] = (radii["DiseaseCategory"] * np.cos(a), radii["DiseaseCategory"] * np.sin(a))
    for s_ in sups:
        a = cm([np.arctan2(P[c][1], P[c][0]) for c in cats if sup_of[c] == s_])
        P[s_] = (radii["SuperCategory"] * np.cos(a), radii["SuperCategory"] * np.sin(a))
    def dis_of(n):
        out = [x for x in nx.all_neighbors(G, n) if NT[x] == "Disease"]
        if NT[n] == "ClinicalRule":
            out += [x for c in nx.all_neighbors(G, n) if NT[c] == "DiseaseCategory" for x in ang if G.has_edge(x, c)]
        return out or list(ang)
    for t in ("ClinicalRule", "Concept"):
        ns = sorted((n for n in G if NT[n] == t), key=lambda n: cm([ang[x] for x in dis_of(n)]) % (2 * np.pi))
        a0 = cm([ang[x] for x in dis_of(ns[0])])
        P.update(ring(ns, radii[t], a0, a0 + 2 * np.pi))
    for g in (n for n in G if NT[n] == "ConceptGroup"):
        a = cm([np.arctan2(P[c][1], P[c][0]) for c in G.predecessors(g) if NT[c] == "Concept"])
        P[g] = (radii["ConceptGroup"] * np.cos(a), radii["ConceptGroup"] * np.sin(a))
    return P, ang


## 1. One node: a disease and its direct neighbors

Concepts on the right (edge width and number = P(concept | disease)), rules on the left, category on top, and a sample of the training images diagnosed with it at the bottom.

In [ ]:
d = f"Disease:{FOCUS_DISEASE}"
concepts = sorted([v for _, v, a in G.out_edges(d, data=True) if a["rel"] == "CHARACTERIZED_BY"],
                  key=lambda v: -G.get_edge_data(d, v)[0]["weight"])
cat = [v for _, v, a in G.out_edges(d, data=True) if a["rel"] == "IS_A"]
rls = [u for u, _, a in G.in_edges(d, data=True) if a["rel"] == "INDICATES"]
imgs_all = [u for u, _, a in G.in_edges(d, data=True) if a["rel"] == "DIAGNOSED_AS" and a["split"] == "train"]
rng = np.random.default_rng(SEED); imgs = list(rng.choice(imgs_all, size=min(12, len(imgs_all)), replace=False))
pos = {d: (0, 0)}
pos.update(ring(concepts, 1.0, np.radians(-40), np.radians(70)))
pos.update(ring(rls, 1.0, np.radians(125), np.radians(200)))
pos.update({c: (0, 1.0) for c in cat})
pos.update(ring(imgs, 0.88, np.radians(-158), np.radians(-62)))
H = sub(set(pos), rels={"CHARACTERIZED_BY", "IS_A", "INDICATES", "DIAGNOSED_AS"})
fig, ax = plt.subplots(figsize=(SINGLE, 3.0))
draw(H, pos, ax, label_types=(), highlight={d}, size_scale=1.4)
for n in H:
    if NT[n] in ("Concept", "ClinicalRule", "DiseaseCategory"):
        txt = short(n) + (f" ({G.get_edge_data(d, n)[0]['weight']:.2f})" if NT[n] == "Concept" else "")
        radial_label(ax, pos[n], txt, off=0.09, fs=6)
ax.text(0, -0.13, FOCUS_DISEASE, ha="center", va="top", fontsize=7, fontweight="bold", color="#D55E00",
        bbox=dict(boxstyle="round,pad=0.08", fc="white", ec="none", alpha=0.85), zorder=6)
ax.text(np.cos(np.radians(-110)) * 1.02, np.sin(np.radians(-110)) * 1.02, f"{len(imgs_all)} training images ({len(imgs)} shown)",
        ha="center", va="top", fontsize=5.5, color="#777777")
legend(ax, ["Disease", "Concept", "ClinicalRule", "DiseaseCategory", "SkinImage"],
       ["CHARACTERIZED_BY", "INDICATES", "IS_A", "DIAGNOSED_AS"], anchor=(0.5, -0.24), ncol=3)
ax.set_xlim(-1.75, 1.95); ax.set_ylim(-1.2, 1.2)
save(fig, "kg01_one_node",
     f"One node of DermKG: '{FOCUS_DISEASE}' and its direct neighbors. Numbers and edge widths give the smoothed P(concept | disease); "
     "the rules indicating the disease, its category and a sample of its training images are also shown.", H)

## 2. Two hops: knowledge neighborhood

Inner ring: direct neighbors. Outer ring: their neighbors (concept groups, premise concepts of the rules, sibling diseases reached through shared rules, the super-category), each placed next to what it connects to.

In [ ]:
d = f"Disease:{FOCUS_DISEASE}"
R2 = {"CHARACTERIZED_BY", "IS_A", "SUBCLASS_OF", "INDICATES", "GOVERNED_BY", "BELONGS_TO"}
hop1_c = sorted([v for _, v, a in G.out_edges(d, data=True) if a["rel"] == "CHARACTERIZED_BY"], key=lambda v: -G.get_edge_data(d, v)[0]["weight"])
hop1_r = sorted(u for u, _, a in G.in_edges(d, data=True) if a["rel"] == "INDICATES")
hop1_k = [v for _, v, a in G.out_edges(d, data=True) if a["rel"] == "IS_A"]
hop1 = hop1_c + hop1_r + hop1_k
hop2 = set()
for n in hop1:
    for _, v, a in G.out_edges(n, data=True):
        if a["rel"] in ("BELONGS_TO", "SUBCLASS_OF", "INDICATES") and v != d:
            hop2.add(v)
    for u, _, a in G.in_edges(n, data=True):
        if a["rel"] == "GOVERNED_BY":
            hop2.add(u)
hop2 -= set(hop1)
pos = {d: (0, 0)}
pos.update(ring(hop1, 1.0, np.radians(-60), np.radians(-60) + 2 * np.pi))
H = sub({d} | set(hop1) | hop2, rels=R2)
def mang(n):
    nb = [x for x in nx.all_neighbors(H, n) if x in hop1]
    a = np.array([np.arctan2(pos[x][1], pos[x][0]) for x in nb]) if nb else np.array([0.0])
    return np.arctan2(np.sin(a).mean(), np.cos(a).mean()) % (2 * np.pi)
h2 = sorted(hop2, key=mang)
a0 = mang(h2[0]) if h2 else 0
pos.update(ring(h2, 2.0, a0, a0 + 2 * np.pi))
fig, ax = plt.subplots(figsize=(DOUBLE * 0.7, 4.3))
draw(H, pos, ax, label_types=(), highlight={d}, size_scale=1.1, edge_alpha=0.45)
for n in H:
    if n != d:
        radial_label(ax, pos[n], short(n).replace("_", " "), off=0.09, fs=5.5)
ax.text(0, -0.13, FOCUS_DISEASE, ha="center", va="top", fontsize=6.5, fontweight="bold", color="#D55E00",
        bbox=dict(boxstyle="round,pad=0.08", fc="white", ec="none", alpha=0.85), zorder=6)
for r_ in (1.0, 2.0):
    ax.add_patch(plt.Circle((0, 0), r_, fill=False, ls=":", lw=0.5, color="#bbbbbb"))
legend(ax, [t for t in COL if any(NT[n] == t for n in H)], sorted({a["rel"] for *_, a in H.edges(data=True)}),
       anchor=(0.5, -0.08), ncol=5)
ax.set_xlim(-2.9, 2.9); ax.set_ylim(-2.45, 2.45)
save(fig, "kg02_two_hops",
     f"Two-hop knowledge neighborhood of '{FOCUS_DISEASE}' ({H.number_of_nodes()} nodes, {H.number_of_edges()} edges). Inner ring: "
     "direct neighbors; outer ring: concept groups, rule premises, sibling diseases reached through shared rules and the super-category.", H)

## 3. One case: a reasoning chain

Image, its concepts, activated rules, indicated diseases (true disease in bold), categories and super-categories, ordered to minimize crossings. A test image whose activated expert rules include its true disease is selected automatically.

In [ ]:
lab = labels.set_index("md5")
cands = []
for m in lab.index[lab.split == "test"]:
    img = f"Image:{m}"
    act = [v for _, v, a in G.out_edges(img, data=True) if a["rel"] == "ACTIVATES"]
    true_d = f"Disease:{lab.at[m, 'label']}"
    hit = [r for r in act if G.has_edge(r, true_d) and G.nodes[r].get("origin") == "expert"]
    if hit and 2 <= len(act) <= 4:
        cands.append((len(act), m))
m = sorted(cands)[-1][1] if cands else lab.index[lab.split == "test"][0]
img, true_d = f"Image:{m}", f"Disease:{lab.at[m, 'label']}"
cons = [v for _, v, a in G.out_edges(img, data=True) if a["rel"] == "HAS_CONCEPT"]
act = [v for _, v, a in G.out_edges(img, data=True) if a["rel"] == "ACTIVATES"]
dis = sorted({v for r in act for _, v, a in G.out_edges(r, data=True) if a["rel"] == "INDICATES" and NT[v] == "Disease"} | {true_d})
dis = dis[:9] if true_d in dis[:9] else dis[:8] + [true_d]
cats = sorted({v for x in dis for _, v, a in G.out_edges(x, data=True) if a["rel"] == "IS_A"})
sups = sorted({v for x in cats for _, v, a in G.out_edges(x, data=True) if a["rel"] == "SUBCLASS_OF"})
H = sub({img} | set(cons + act + dis + cats + sups), rels={"HAS_CONCEPT", "ACTIVATES", "GOVERNED_BY", "INDICATES", "IS_A", "SUBCLASS_OF"})
pos = layered(H, [[img], cons, act, dis, cats, sups], spacing=0.42)
fig, ax = plt.subplots(figsize=(DOUBLE, 0.9 + 0.42 * max(len(L) for L in [cons, act, dis])))
draw(H, pos, ax, label_types=(), highlight={true_d}, size_scale=1.5)
ax.scatter(*pos[img], s=120, c=COL["SkinImage"], edgecolors="black", lw=0.6, zorder=4)
for n in H:
    txt = f"image {m[:6]}" if n == img else short(n) + (f"\n({G.nodes[n].get('strength', '')})" if NT[n] == "ClinicalRule" else "")
    ax.text(pos[n][0], pos[n][1] + 0.09, txt, ha="center", va="bottom", fontsize=6, zorder=5,
            color="#D55E00" if n == true_d else "black", fontweight="bold" if n == true_d else "normal",
            bbox=dict(boxstyle="round,pad=0.04", fc="white", ec="none", alpha=0.75))
ax.set_aspect("auto")
top = max(p[1] for p in pos.values()) + 0.45
for i, t in enumerate(["Image", "Concepts", "Activated rules", "Indicated diseases", "Categories", "Super-categories"]):
    ax.text(i, top, t, ha="center", fontsize=7, fontweight="bold")
ax.set_ylim(min(p[1] for p in pos.values()) - 0.15, top + 0.15)
save(fig, "kg03_one_case",
     f"Reasoning chain for one test image (true diagnosis in bold: {lab.at[m, 'label']}). Annotated concepts activate clinical rules, "
     "which indicate candidate diseases that roll up along IS_A and SUBCLASS_OF.", H)

## 4. One category

Layered: concepts, expert rules, diseases, category.

In [ ]:
cat = f"Category:{FOCUS_CATEGORY}"
dis = sorted(u for u, _, a in G.in_edges(cat, data=True) if a["rel"] == "IS_A")
cons = sorted({v for x in dis for _, v, a in G.out_edges(x, data=True) if a["rel"] == "CHARACTERIZED_BY"})
rls = {u for x in dis for u, _, a in G.in_edges(x, data=True) if a["rel"] == "INDICATES" and G.nodes[u].get("origin") == "expert"}
rls |= {u for u, _, a in G.in_edges(cat, data=True) if a["rel"] == "INDICATES"}
rls = sorted(rls)
H = sub({cat} | set(dis) | set(cons) | set(rls), rels={"IS_A", "CHARACTERIZED_BY", "INDICATES", "GOVERNED_BY"})
pos = layered(H, [cons, rls, dis, [cat]], height=2.0)
pos[cat] = (4.4, 0.0)
fig, ax = plt.subplots(figsize=(SINGLE, 3.6))
draw(H, pos, ax, label_types=(), highlight={cat}, edge_alpha=0.45, size_scale=1.0)
for n in H:
    x, y = pos[n]
    if x == 0:
        ax.text(x - 0.08, y, short(n), ha="right", va="center", fontsize=5.5)
    elif NT[n] == "ClinicalRule":
        ax.text(x, y + 0.07, short(n), ha="center", va="bottom", fontsize=5.5, color="#009E73")
    elif NT[n] == "DiseaseCategory":
        ax.text(x, y + 0.13, short(n).replace(" ", "\n"), ha="center", va="bottom", fontsize=5.5)
    else:
        ax.text(x + 0.08, y, short(n), ha="left", va="center", fontsize=5.2,
                bbox=dict(boxstyle="round,pad=0.04", fc="white", ec="none", alpha=0.8), zorder=5)
ax.set_aspect("auto"); ax.set_xlim(-0.9, 5.0); ax.set_ylim(-1.1, 1.3)
for x_, t in zip([0, 1, 2, 4.4], ["Concepts", "Rules", "Diseases", "Category"]):
    ax.text(x_, 1.2, t, ha="center", fontsize=6.5, fontweight="bold")
save(fig, f"kg04_category_{FOCUS_CATEGORY.replace(' ', '_')}",
     f"The '{FOCUS_CATEGORY}' category in DermKG: {len(dis)} diseases with their characteristic concepts (CHARACTERIZED_BY) and the "
     f"{len(rls)} expert rules that point to them (premises via GOVERNED_BY, conclusions via INDICATES).", H)

## 5. One super-category

Layered: concepts, expert rules, diseases, categories, super-category.

In [ ]:
sup = f"Super:{FOCUS_SUPER}"
cats = sorted(u for u, _, a in G.in_edges(sup, data=True) if a["rel"] == "SUBCLASS_OF")
dis = sorted(u for c in cats for u, _, a in G.in_edges(c, data=True) if a["rel"] == "IS_A")
cons = sorted({v for x in dis for _, v, a in G.out_edges(x, data=True) if a["rel"] == "CHARACTERIZED_BY"})
rls = sorted({u for x in dis + cats for u, _, a in G.in_edges(x, data=True) if a["rel"] == "INDICATES" and G.nodes[u].get("origin") == "expert"})
H = sub({sup} | set(cats + dis + cons + rls), rels={"SUBCLASS_OF", "IS_A", "CHARACTERIZED_BY", "INDICATES", "GOVERNED_BY"})
dis = sorted(dis, key=lambda x: (G.nodes[x]["category"], x))
cats = sorted(cats)
pos = layered(H, [cons, rls, dis, cats, [sup]], height=2.4, fixed=(2, 3, 4))
pos.update({n: (x + (1.2 if x >= 3 else 0), y) for n, (x, y) in pos.items()})
fig, ax = plt.subplots(figsize=(DOUBLE, 3.6))
draw(H, pos, ax, label_types=(), highlight={sup}, edge_alpha=0.4, size_scale=1.0)
for n in H:
    x, y = pos[n]
    if x == 0:
        ax.text(x - 0.07, y, short(n), ha="right", va="center", fontsize=5.5)
    elif NT[n] == "ClinicalRule":
        ax.text(x, y + 0.06, short(n), ha="center", va="bottom", fontsize=5.5, color="#009E73")
    else:
        ax.text(x + 0.07, y, short(n), ha="left", va="center", fontsize=5.5,
                bbox=dict(boxstyle="round,pad=0.04", fc="white", ec="none", alpha=0.8), zorder=5)
ax.set_aspect("auto"); ax.set_xlim(-0.8, 6.4); ax.set_ylim(-1.35, 1.5)
for x_, t in zip([0, 1, 2, 4.2, 5.2], ["Concepts", "Rules", "Diseases", "Categories", "Super-category"]):
    ax.text(x_, 1.38, t, ha="center", fontsize=6.5, fontweight="bold")
legend(ax, ["SuperCategory", "DiseaseCategory", "Disease", "ClinicalRule", "Concept"],
       ["SUBCLASS_OF", "IS_A", "CHARACTERIZED_BY", "INDICATES", "GOVERNED_BY"], anchor=(0.5, -0.1), ncol=5)
save(fig, f"kg05_super_{FOCUS_SUPER}",
     f"The '{FOCUS_SUPER}' branch of DermKG: {len(cats)} categories, {len(dis)} diseases, {len(cons)} concepts and {len(rls)} expert rules.", H)

## 6. Knowledge layer (all non-image nodes)

Radial layout from the inside out: super-categories, categories, diseases (in category sectors), rules, concepts, concept groups. Rules and concepts sit at the mean angle of the diseases they connect to.

In [ ]:
RAD6 = {"SuperCategory": 0.75, "DiseaseCategory": 1.95, "Disease": 3.0, "ClinicalRule": 4.2, "Concept": 4.9, "ConceptGroup": 5.6}
pos6, ang6 = sector_positions(RAD6)
REL6 = {"SUBCLASS_OF", "IS_A", "CHARACTERIZED_BY", "INDICATES", "GOVERNED_BY", "BELONGS_TO"}
H = sub(set(pos6), rels=REL6)
SIZE_BAK = dict(SIZE); SIZE.update({"Disease": 14, "Concept": 26, "ClinicalRule": 18, "DiseaseCategory": 90, "SuperCategory": 140, "ConceptGroup": 60})
fig, ax = plt.subplots(figsize=(DOUBLE, DOUBLE))
draw(H, pos6, ax, label_types=(), edge_alpha=0.13, width_scale=0.6)
SIZE.clear(); SIZE.update(SIZE_BAK)
for x, a in ang6.items():                         # disease labels pointing outward, between diseases and rules
    deg = np.degrees(a); flip = 90 < deg % 360 < 270
    p = np.array(pos6[x]) * 1.025
    ax.text(*p, short(x), fontsize=3.4, rotation=deg + (180 if flip else 0), rotation_mode="anchor",
            ha="right" if flip else "left", va="center", zorder=5, color="#7a2e00")
for n in H:
    if NT[n] in ("Concept", "ConceptGroup"):
        radial_label(ax, pos6[n], short(n).replace("_", " "), off=0.07, fs=5 if NT[n] == "Concept" else 6)
    elif NT[n] in ("DiseaseCategory", "SuperCategory"):
        ax.text(pos6[n][0], pos6[n][1] - 0.12, short(n), fontsize=5.2, ha="center", va="top", zorder=6,
                bbox=dict(boxstyle="round,pad=0.05", fc="white", ec="none", alpha=0.85))
legend(ax, ["SuperCategory", "DiseaseCategory", "Disease", "ClinicalRule", "Concept", "ConceptGroup"], sorted(REL6),
       anchor=(0.5, -0.03), ncol=6)
ax.set_xlim(-6.4, 6.4); ax.set_ylim(-6.2, 6.2)
save(fig, "kg06_knowledge_layer",
     f"The DermKG knowledge layer ({H.number_of_nodes()} nodes, {H.number_of_edges()} edges; image nodes omitted). From the center outward: "
     "super-categories, categories, diseases in category sectors, clinical rules, concepts and concept groups. Rules and concepts are placed "
     "at the mean angle of the diseases they connect to.", H)

## 7. Whole graph (all nodes, including images)

Same radial frame as panel 6 with the 3,230 image nodes added as a band just outside their diagnoses, so band density shows the class distribution. Drawn edges: knowledge edges plus `DIAGNOSED_AS`; the image-to-concept, image-to-rule and image-to-skin-type edges are counted in the caption but not drawn.

In [ ]:
RAD7 = {"SuperCategory": 0.75, "DiseaseCategory": 1.95, "Disease": 3.0, "ClinicalRule": 4.3, "Concept": 4.95, "ConceptGroup": 5.6}
pos7, ang7 = sector_positions(RAD7)
rng = np.random.default_rng(SEED)
img_dis = {u: v for u, v, a in G.edges(data=True) if a["rel"] == "DIAGNOSED_AS"}
step = 2 * np.pi / (len(ang7) + 2 * len([n for n in G if NT[n] == "DiseaseCategory"]))
for im, dz in img_dis.items():
    a = ang7[dz] + rng.uniform(-0.42, 0.42) * step
    r = 3.15 + rng.gamma(2.0, 0.12)
    pos7[im] = (r * np.cos(a), r * np.sin(a))
fig, ax = plt.subplots(figsize=(DOUBLE, DOUBLE))
segs = np.array([[pos7[u], pos7[v]] for u, v in img_dis.items()])
ax.add_collection(mpl.collections.LineCollection(segs, colors=ECOL["DIAGNOSED_AS"], linewidths=0.2, alpha=0.25, zorder=1))
know = [(u, v, a) for u, v, a in G.edges(data=True) if NT[u] not in ("SkinImage", "SkinType") and NT[v] not in ("SkinImage", "SkinType") and a["rel"] != "CO_OCCURS"]
for rel in sorted({a["rel"] for *_, a in know}):
    seg = np.array([[pos7[u], pos7[v]] for u, v, a in know if a["rel"] == rel])
    ax.add_collection(mpl.collections.LineCollection(seg, colors=ECOL[rel], linewidths=0.3, alpha=0.18, zorder=1))
xy = np.array([pos7[n] for n in img_dis]); ax.scatter(xy[:, 0], xy[:, 1], s=0.8, c="#888888", lw=0, zorder=2)
sz = {"Disease": 10, "Concept": 22, "ClinicalRule": 14, "DiseaseCategory": 80, "SuperCategory": 120, "ConceptGroup": 50}
for t, s_ in sz.items():
    q = np.array([pos7[n] for n in G if NT[n] == t])
    ax.scatter(q[:, 0], q[:, 1], s=s_, c=COL[t], marker=SHAPE.get(t, "o"), edgecolors="white", lw=0.3, zorder=3)
for n in G:
    if NT[n] in ("DiseaseCategory", "SuperCategory"):
        ax.text(pos7[n][0], pos7[n][1] - 0.12, short(n), fontsize=5.2, ha="center", va="top", zorder=6,
                bbox=dict(boxstyle="round,pad=0.05", fc="white", ec="none", alpha=0.85))
    elif NT[n] in ("Concept", "ConceptGroup"):
        radial_label(ax, pos7[n], short(n).replace("_", " "), off=0.07, fs=4.6 if NT[n] == "Concept" else 5.5)
ax.set_aspect("equal"); ax.axis("off"); ax.set_xlim(-6.4, 6.4); ax.set_ylim(-6.2, 6.2)
legend(ax, ["SkinImage", "Disease", "ClinicalRule", "Concept", "ConceptGroup", "DiseaseCategory", "SuperCategory"], anchor=(0.5, -0.02), ncol=7)
n_img_edges = sum(1 for u, v, a in G.edges(data=True) if a["rel"] in ("HAS_CONCEPT", "ACTIVATES", "HAS_SKIN_TYPE"))
save(fig, "kg07_whole_graph",
     f"The complete DermKG ({G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges). Gray dots are the {len(img_dis):,} images, "
     "placed as a band beside their diagnosed disease, so band density reflects the long-tailed class distribution. Knowledge edges and "
     f"DIAGNOSED_AS are drawn; the {n_img_edges:,} image-to-concept, image-to-rule and image-to-skin-type edges are omitted for readability.")

## 8. Zoom strip for the paper (one figure, four panels)

Re-draws panels 1, 3, 5 and 6 at small size in one row. Run cells 1, 3, 5, 6 first.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(DOUBLE, 2.2), gridspec_kw={"width_ratios": [1, 1.5, 1, 1]})
def redraw(ax, name):
    ax.imshow(plt.imread(os.path.join(FIG_DIR, f"{name}.png"))); ax.axis("off")
names = ["kg01_one_node", "kg03_one_case", f"kg05_super_{FOCUS_SUPER}", "kg06_knowledge_layer"]
for ax, n, t in zip(axes, names, ["(a) One node", "(b) One case", "(c) One branch", "(d) Knowledge layer"]):
    redraw(ax, n); ax.set_title(t, fontsize=7, y=-0.12)
plt.subplots_adjust(wspace=0.02)
FORMATS_BAK = FORMATS; FORMATS = ("png", "pdf")
save(fig, "kg08_zoom_strip",
     "DermKG at increasing scale: (a) a single disease node and its neighbors, (b) the reasoning chain of one image, "
     f"(c) the {FOCUS_SUPER} branch, (d) the full knowledge layer.")
FORMATS = FORMATS_BAK
print("note: this strip is assembled from the PNGs; for a fully vector version, place the individual PDFs as subfigures in LaTeX.")

## Index

In [ ]:
print(open(os.path.join(FIG_DIR, "captions.md")).read())
print(sorted(os.listdir(FIG_DIR)))